# 03 · Preparação para a modelagem

**CRISP-DM fase 3 — preparação dos dados** · Retenção de clientes Telecom

---

O notebook [`02-ajustes-dados`](02-ajustes-dados.ipynb) entregou a **base curada**: 7.043
clientes, 19 colunas, nenhuma célula vazia e cada ajuste com motivo registrado.  Ela descreve
a carteira em **unidades de negócio**: meses de casa, valores em dólar, respostas `Yes/No`.
É justamente isso que a torna legível para a empresa e **imprópria para um algoritmo
baseado em distância**, como o K-Means usado nos notebooks 04.

O algoritmo só entende números, e compara clientes medindo a distância entre eles. Por
isso este notebook faz três coisas:

1. **Seleciona** as colunas que entram na segmentação (e justifica as que ficam de fora),
   criando as variáveis derivadas sugeridas pela EDA;
2. **Codifica** as categorias em números (`Yes/No` → `1/0`);
3. **Coloca tudo na mesma escala** (0 a 1), para nenhuma coluna pesar mais que as outras
   só por ter números maiores.

In [ ]:
# ============================================================================
# Configuração de avisos
# ============================================================================
import warnings

def warn(*args, **kwargs):
    pass

warnings.warn = warn
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

# ============================================================================
# Importações
# ============================================================================
from itertools import combinations

import joblib
import numpy as np
import pandas as pd
from scipy.stats.contingency import association
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# ============================================================================
# Escalonamento de variáveis numéricas
# ============================================================================
# MinMaxScaler    → ajusta cada variável (coluna) para o intervalo de 0 a 1.
#
# StandardScaler  → centraliza cada variável (coluna) em média 0 e desvio
#                   padrão 1. Não torna a distribuição normal.
#
# Normalizer      → ajusta cada observação (linha) para norma L2 igual a 1,
#                   ou seja, a soma dos quadrados dos valores da linha é 1.

# ============================================================================
# Codificação de variáveis categóricas
# ============================================================================
# OneHotEncoder   → cria uma coluna binária (0/1) para cada categoria.
#                   Indicado para variáveis nominais.
#
# OrdinalEncoder  → converte cada categoria em um número inteiro. A ordem só
#                   é respeitada se informada em `categories`; caso contrário,
#                   segue a ordem alfabética. Indicado para variáveis ordinais.

# Carregando a base curada

In [ ]:
# Base curada do notebook 02 (linhas na mesma ordem da base bruta)
df = pd.read_csv('../data/processed/Telcon-Customer-Churn_tratado.csv', sep=',')

# Registro das decisões sobre cada coluna
decisoes = []

def decidir(coluna, decisao, motivo):
    decisoes.append({'coluna': coluna, 'decisao': decisao, 'motivo': motivo})

df.shape

# 1. Seleção das colunas

Nem toda coluna da base deve entrar na segmentação. Uma coluna fica de fora quando:

- **repete** a informação de outras colunas (contaria a mesma coisa duas vezes);
- **não se relaciona com nada** (só acrescenta ruído à distância entre clientes).

`Churn` já não está na base: foi retirada no notebook 02, porque o projeto é não
supervisionado.

## 1.1 `TotalCharges`: informação repetida

No notebook 02 vimos que `TotalCharges ≈ tenure × MonthlyCharges`. Ou seja, ela é
**calculada** a partir de duas colunas que já estão na base. A correlação confirma:

In [ ]:
numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[numericas].corr(method='spearman').round(2)

`TotalCharges` tem correlação de **0,89** com `tenure`: um cliente com muitos meses de casa
quase sempre tem total alto. Se ela entrasse, o "tempo de casa" seria contado **duas
vezes** na distância entre clientes, e os grupos acabariam separados principalmente por
antiguidade.

Bônus: ela era a única variável numérica assimétrica (assimetria ~0,96, notebook 02). Ao
retirá-la, não precisamos aplicar transformação logarítmica.

In [ ]:
print(df[numericas].skew().round(2))

decidir('TotalCharges', 'Fora',
        'Derivada de tenure × MonthlyCharges (Spearman 0,89 com tenure); contaria o tempo de casa duas vezes')

## 1.2 `gender`: sem relação com nenhuma outra coluna

Calculamos o **V de Cramér** (força da associação entre duas categorias, de 0 a 1) para
todos os pares de colunas categóricas **da base curada do notebook 02**: já sem `Churn` e
com as categorias estruturais recodificadas. É a mesma medida da análise multivariada do
notebook 01, agora sobre os dados que de fato vão para o modelo. Vamos ver como `gender`
se sai:

In [ ]:
# V de Cramér de todos os pares de colunas categóricas da base curada
# (Churn já foi retirada no notebook 02: nenhuma decisão é tomada olhando para ela)
colunas_categoricas = df.select_dtypes(exclude='number').columns.tolist()
associacoes = pd.DataFrame([
    {'var_1': a, 'var_2': b,
     'v_cramer': association(pd.crosstab(df[a], df[b]), method='cramer')}
    for a, b in combinations(colunas_categoricas, 2)
]).sort_values('v_cramer', ascending=False)

mais_associado = associacoes.iloc[0]
print(f'Par mais associado da base: {mais_associado.var_1} × {mais_associado.var_2} '
      f'(V = {mais_associado.v_cramer:.2f})')

pares_gender = associacoes.query('var_1 == "gender" or var_2 == "gender"')
print(f'V de Cramér máximo de gender com as outras {len(pares_gender)} colunas: '
      f'{pares_gender["v_cramer"].max():.3f}')
pares_gender.head()

O maior V de Cramér de `gender` é **0,02**, praticamente zero. Para comparar, o par mais
associado da base (`StreamingTV` × `StreamingMovies`) tem 0,53. Homens e mulheres
contratam os mesmos serviços, nos mesmos contratos e pagam do mesmo jeito.

Por que isso atrapalha? `gender` divide a base quase meio a meio. Para o K-Means, dois
clientes idênticos em tudo, exceto no gênero, ficariam a uma distância de 1 (o máximo
de uma coluna binária). O algoritmo poderia "descobrir" grupos de homens e mulheres que
**não diferem em nada** de útil para a retenção.

In [ ]:
print(df['gender'].value_counts(normalize=True).round(3))

decidir('gender', 'Fora',
        'V de Cramér máximo 0,02 com as demais colunas: não estrutura nada, só acrescenta ruído')

## 1.3 Variáveis derivadas (sugestões da EDA)

Os insights do notebook 01 sugeriram duas mudanças. Elas vieram de leituras da EDA, mas
aqui **cada uma é justificada sem olhar para `Churn`**, para manter a regra de que nenhuma
decisão de preparação usa o rótulo.

### a) Contagem de serviços de proteção

`OnlineSecurity`, `OnlineBackup`, `DeviceProtection` e `TechSupport` são os quatro serviços
que protegem a conexão do cliente, e eles andam juntos: quem contrata um tende a contratar
outro.

In [ ]:
servicos_protecao = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']

# Associação entre os quatro serviços de proteção
pd.DataFrame([
    {'var_1': a, 'var_2': b,
     'v_cramer': round(association(pd.crosstab(df[a], df[b]), method='cramer'), 2)}
    for a, b in combinations(servicos_protecao, 2)
])

Todos os pares têm V de Cramér entre 0,28 e 0,35. Como quatro colunas `0/1`, a proteção
entraria **quatro vezes** na distância entre clientes, o mesmo problema da seção 1.1. Elas
são resumidas em **uma** variável, `ServicosProtecao`: quantos dos quatro serviços o
cliente contrata (0 a 4). O "quanto o cliente está protegido" continua no modelo, com o
peso de uma coluna só. As quatro colunas originais seguem na base curada do notebook 02,
para descrever os segmentos depois.

### b) Pagamentos automáticos agrupados

`Bank transfer (automatic)` e `Credit card (automatic)` são o mesmo comportamento: o
cliente não precisa agir para pagar. Os perfis dos dois grupos confirmam:

In [ ]:
# Perfil de cada forma de pagamento (sem Churn)
perfil_pagamento = pd.concat([
    df.groupby('PaymentMethod')[['tenure', 'MonthlyCharges']].median(),
    pd.crosstab(df['PaymentMethod'], df['Contract'], normalize='index').round(2),
    pd.crosstab(df['PaymentMethod'], df['PaperlessBilling'], normalize='index')['Yes']
      .round(2).rename('PaperlessBilling_Yes'),
], axis=1)
perfil_pagamento

Os dois automáticos são praticamente idênticos (`tenure` mediano 48 × 47, mensalidade
73 × 73, mesma divisão de contratos e de fatura digital), enquanto `Electronic check` e
`Mailed check` têm perfis bem diferentes entre si e dos automáticos. Manter os dois
automáticos separados criaria uma coluna que não distingue nada. Eles viram uma só
categoria, `Automatic`.

As duas transformações ficam numa função, para serem repetidas **igualzinho** na
implantação (notebook 06), antes do pré-processador.

In [ ]:
def derivar_variaveis(dados):
    """Cria as variáveis derivadas da seção 1.3 a partir da base curada do notebook 02."""
    dados = dados.copy()

    # a) Quantos dos quatro serviços de proteção o cliente contrata (0 a 4)
    dados['ServicosProtecao'] = dados[servicos_protecao].eq('Yes').sum(axis=1)
    dados = dados.drop(columns=servicos_protecao)

    # b) Débito e cartão automáticos viram uma só categoria
    automaticos = {'Bank transfer (automatic)': 'Automatic',
                   'Credit card (automatic)': 'Automatic'}
    dados['PaymentMethod'] = dados['PaymentMethod'].replace(automaticos)
    return dados


df = derivar_variaveis(df)

decidir('ServicosProtecao', 'Criada',
        'Contagem (0 a 4) de OnlineSecurity, OnlineBackup, DeviceProtection e TechSupport')
for c in servicos_protecao:
    decidir(c, 'Fora', 'Resumida em ServicosProtecao (V de Cramér 0,28 a 0,35 entre os quatro)')
decidir('PaymentMethod', 'Agrupada',
        'Bank transfer e Credit card (automatic) viram Automatic: perfis idênticos')

print(df['PaymentMethod'].value_counts().to_string())
df['ServicosProtecao'].value_counts().sort_index()

## 1.4 Colunas que entram

Todas as outras entram. Separamos por tipo, porque cada tipo recebe um tratamento:

- **Numéricas:** `tenure`, `MonthlyCharges`, `ServicosProtecao`;
- **Binárias** (só duas respostas, `Yes/No`): viram **uma** coluna `0/1`;
- **Com 3 ou mais categorias** (`InternetService`, `Contract`, `PaymentMethod`): viram uma
  coluna `0/1` **para cada categoria**.

In [ ]:
fora = ['TotalCharges', 'gender']
colunas = [c for c in df.columns if c not in fora]

numericas = ['tenure', 'MonthlyCharges', 'ServicosProtecao']
categoricas = [c for c in colunas if c not in numericas]
binarias = [c for c in categoricas if df[c].nunique() == 2]
multicategoricas = [c for c in categoricas if df[c].nunique() > 2]

for c in numericas:
    decidir(c, 'Entra', 'Numérica: escalonada para 0 a 1')
for c in binarias:
    decidir(c, 'Entra', 'Binária: uma coluna 0/1')
for c in multicategoricas:
    decidir(c, 'Entra', f'{df[c].nunique()} categorias: uma coluna 0/1 por categoria')

print('Numéricas      :', numericas)
print('Binárias       :', binarias)
print('Multicategorias:', multicategoricas)

# 2. Codificação e escalonamento

## Por que escalonar?

Veja a amplitude das duas colunas numéricas:

In [ ]:
df[numericas].describe().loc[['min', 'max']]

`tenure` vai de 0 a 72, `MonthlyCharges` de 18 a 119, `ServicosProtecao` de 0 a 4, e as
colunas binárias vão de 0 a 1.
Numa distância, uma diferença de 50 dólares na mensalidade "valeria" 50 vezes mais que
ter ou não ter um serviço. O `MinMaxScaler` leva as numéricas para **0 a 1**, a mesma
faixa das colunas `0/1`. Assim, todas as colunas começam com o mesmo peso.

A fórmula é simples:

$$x_{novo} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

Um cliente com `tenure = 36` (metade de 72) fica com `0,5`.

## Por que one-hot?

Não dá para transformar `Contract` em `1, 2, 3`: o algoritmo entenderia que
`Two year` (3) está "mais longe" de `Month-to-month` (1) do que `One year` (2), e que
existe uma ordem matemática entre eles. O **one-hot** cria uma coluna por categoria,
com `1` na categoria do cliente e `0` nas outras, sem inventar ordem.

Para as binárias, `drop='if_binary'` guarda **só uma** das duas colunas: `Partner_Yes`
já diz tudo (se for 0, o cliente não tem parceiro). Duas colunas contariam a mesma
informação duas vezes, o mesmo problema da seção 1.

In [ ]:
# Pipelines de pré-processamento para cada tipo de variável
## Numéricas
numeric_preprocessor = Pipeline(steps=[
    ('normalization', MinMaxScaler())  # Normalização para 0 a 1
])

## Categóricas (binárias viram uma coluna; as demais, uma por categoria)
categorical_preprocessor = Pipeline(steps=[
    ('encoder', OneHotEncoder(drop='if_binary', sparse_output=False, dtype=int)),
])

# Colunas fora da lista (TotalCharges, gender) são descartadas (remainder='drop')
preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_preprocessor, numericas),
    ('categorical', categorical_preprocessor, categoricas),
], verbose_feature_names_out=False).set_output(transform='pandas')

In [ ]:
# Aplicando o pré-processamento
X_preprocessed = preprocessor.fit_transform(df)
X_preprocessed.shape

# 3. Conferindo o resultado

Sempre confira o que saiu: é aqui que erros de codificação aparecem.

In [ ]:
# Antes e depois para os dois primeiros clientes
pd.concat({'antes': df.head(2).T, 'depois': X_preprocessed.head(2).T}, axis=1).head(12)

In [ ]:
# Todas as colunas devem estar entre 0 e 1, sem vazios
assert X_preprocessed.notna().all().all()
assert X_preprocessed.min().min() >= 0 and X_preprocessed.max().max() <= 1
assert len(X_preprocessed) == len(df)

X_preprocessed.describe().T[['mean', 'min', 'max']].round(2)

A coluna `mean` das variáveis `0/1` é a **proporção de clientes** com aquela
característica. Por exemplo, a média de `Contract_Month-to-month` é a fração de clientes
com contrato mensal. Confira se os números batem com o que a EDA mostrou.

# Registro das decisões

In [ ]:
tabela_decisoes = pd.DataFrame(decisoes)
tabela_decisoes.to_csv('../docs/tabelas/preparacao-colunas.csv', index=False)
tabela_decisoes

# Salvando a base pré-processada

São gravados dois arquivos:

- `data/processed/Telcon-Customer-Churn_preprocessed.csv`: a base pronta para os notebooks 04, com as
  linhas na mesma ordem da base curada (e da bruta). Leitura:
  ```python
  X = pd.read_csv('../data/processed/Telcon-Customer-Churn_preprocessed.csv', sep=',')
  ```
- `models/preprocessador.joblib`: o pré-processador **já ajustado** (com os mínimos e
  máximos desta base). Na implantação (notebook 06), um cliente novo precisa passar
  **exatamente** pela mesma transformação, e é este arquivo que garante isso. Antes dele,
  o cliente novo passa pela função `derivar_variaveis` (seção 1.3).

In [ ]:
X_preprocessed.to_csv('../data/processed/Telcon-Customer-Churn_preprocessed.csv', sep=',', index=False)
joblib.dump(preprocessor, '../models/preprocessador.joblib')
print(f'{X_preprocessed.shape[0]} clientes, {X_preprocessed.shape[1]} colunas gravadas')